# Customer Segmentation — Clustering Analysis

This notebook combines the methodology presented in class with the extended clustering workflow from the reference project.

The workflow includes:

1. Loading the cleaned customer dataset  
2. Comparing scaling methods  
3. Evaluating K-Means  
4. Applying Ward Hierarchical Clustering  
5. Comparing K-Means and Ward solutions  
6. Selecting the final clustering solution  
7. Profiling and naming customer segments  
8. Visualizing clusters with PCA  
9. Exporting the clustered dataset  


## 1. Imports

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

import utils_clustering as utils

ModuleNotFoundError: No module named 'utils_clustering'

## 2. Load Cleaned Dataset

In [ ]:
path = "/Users/franciscateixeira/Documents/ML/CostumerSegP_MLII/datasets/customer_info_clean.csv"

df = pd.read_csv(path, index_col="customer_id")

utils.validate_clustering_data(df)

## 3. Create Scaled Versions

Clustering algorithms are distance-based, so scaling is essential.

We compare:

- StandardScaler
- MinMaxScaler
- RobustScaler

The unscaled dataset is not used as a final solution because the variables have very different scales.


In [ ]:
scaled_versions, fitted_scalers = utils.create_scaled_versions(
    df,
    include_no_scaling=False
)

print(scaled_versions.keys())

## 4. K-Means Evaluation

K-Means is evaluated for different values of `k`.

The following metrics are calculated:

- Inertia
- Silhouette Score
- Davies-Bouldin Score
- Calinski-Harabasz Score


In [ ]:
kmeans_results = utils.evaluate_kmeans_for_scalers(
    scaled_versions,
    k_range=range(2, 11),
    random_state=42
)

## 5. Elbow Curve and Silhouette Score

In [ ]:
utils.plot_kmeans_metrics(kmeans_results)

## 6. Ward Hierarchical Clustering — Dendrogram

Ward linkage is used as the hierarchical clustering method because it minimizes within-cluster variance.

A sample is used for the dendrogram to avoid excessive computation and unreadable plots.


In [ ]:
X_standard = scaled_versions["standard"]

linked = utils.plot_ward_dendrogram(
    X_standard,
    sample_size=3000,
    random_state=42,
    truncate_level=5
)

## 7. Choose Final Solution

Adjust `final_scaler_name` and `best_k` after inspecting:

- Elbow curve
- Silhouette score
- Dendrogram
- Business interpretability


In [ ]:
final_scaler_name = "standard"
best_k = 5

X_final = scaled_versions[final_scaler_name]

print(f"Final scaler: {final_scaler_name}")
print(f"Final number of clusters: {best_k}")

## 8. Fit Final K-Means Solution

In [ ]:
kmeans_final, kmeans_labels = utils.fit_kmeans_final(
    X_final,
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

df_clustered = df.copy()
df_clustered["cluster_kmeans"] = kmeans_labels

utils.plot_cluster_sizes(df_clustered, "cluster_kmeans")

## 9. Fit Ward Solution with the Same Number of Clusters

In [ ]:
ward_final, ward_labels = utils.fit_ward_clustering(
    X_final,
    n_clusters=best_k
)

df_clustered["cluster_ward"] = ward_labels

utils.plot_cluster_sizes(df_clustered, "cluster_ward")

## 10. Compare K-Means vs Ward

This comparison checks whether both clustering approaches identify similar customer structures.


In [ ]:
comparison = utils.compare_cluster_solutions(
    df_clustered["cluster_kmeans"],
    df_clustered["cluster_ward"],
    name_1="K-Means",
    name_2="Ward"
)

## 11. Choose Final Cluster Column

K-Means is usually easier to explain and reproduce.

Ward is used as a validation method. If Ward gives more interpretable segments, change `final_cluster_col` to `"cluster_ward"`.


In [ ]:
final_cluster_col = "cluster_kmeans"

df_clustered["final_cluster"] = df_clustered[final_cluster_col]

## 12. Cluster Profiling

Each cluster is profiled by comparing its feature averages against the global average.

Interpretation:

- Values above 1 indicate that the cluster is above the global average.
- Values below 1 indicate that the cluster is below the global average.


In [ ]:
original_features = df.columns.tolist()

cluster_means = utils.calculate_cluster_means(
    df_clustered,
    "final_cluster"
)

display(cluster_means)

cluster_profile = utils.calculate_cluster_profile(
    df_clustered,
    "final_cluster",
    original_features
)

display(cluster_profile.T)

utils.plot_cluster_profile_heatmap(cluster_profile)

## 13. Top Drivers per Cluster

In [ ]:
for cluster_id in sorted(df_clustered["final_cluster"].unique()):
    print(f"\nCluster {cluster_id} — Top differentiating features")
    display(
        utils.show_top_cluster_drivers(
            cluster_profile,
            cluster_id,
            top_n=10
        )
    )

## 14. Name Clusters

Update the mapping below after analyzing the cluster profiles.

Use professional names, such as:

- High Value Customers
- Promotion-Sensitive Customers
- Technology-Oriented Customers
- Family-Oriented Customers
- Pet-Oriented Customers
- Service-Sensitive Customers
- Low Engagement Customers
- Loyal Premium Customers


In [ ]:
cluster_names = {
    0: "Segment 0",
    1: "Segment 1",
    2: "Segment 2",
    3: "Segment 3",
    4: "Segment 4"
}

df_clustered["cluster_name"] = df_clustered["final_cluster"].map(cluster_names)

display(df_clustered["cluster_name"].value_counts())

## 15. PCA Visualization

In [ ]:
pca_df, pca_model = utils.plot_pca_clusters(
    X_final,
    df_clustered["final_cluster"],
    title="Customer Segments - PCA Projection"
)

## 16. Optional UMAP Visualization

Only run this cell if `umap-learn` is installed.

If it is not installed, PCA is enough for the report.


In [ ]:
# umap_df, umap_model = utils.plot_umap_clusters(
#     X_final,
#     df_clustered["final_cluster"],
#     title="Customer Segments - UMAP Projection"
# )

## 17. Export Final Clustered Dataset

In [ ]:
output_path = "/Users/franciscateixeira/Documents/ML/CostumerSegP_MLII/datasets/customer_info_clustered.csv"

utils.export_clustered_dataset(
    df_clustered,
    output_path
)

## 18. Methodology Text for Report

In [ ]:
methodology_text = """
After preprocessing, the cleaned customer dataset was used for clustering.
Since clustering algorithms are distance-based, different scaling methods were
compared: StandardScaler, MinMaxScaler and RobustScaler.

K-Means was evaluated for different numbers of clusters using the elbow method,
based on inertia, and the silhouette score. Additional validation metrics such
as Davies-Bouldin and Calinski-Harabasz were also calculated.

Hierarchical clustering using Ward linkage was applied as a complementary
method. A dendrogram was generated on a sample of the data to inspect the
hierarchical structure, and the final Ward solution was compared against the
K-Means solution.

The final clustering solution was selected based on quantitative metrics,
visual inspection and business interpretability. The resulting clusters were
profiled by comparing each cluster's feature averages against the global
average, allowing the segments to be assigned meaningful business names.
"""

print(methodology_text)